In [42]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [43]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [44]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [45]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [46]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [47]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [48]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [49]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3322, grad_fn=<NegBackward0>)

In [50]:
# so loss is mean of values picked from logprob, loss = -(whatever values were picked because of Yb)/32
# hence differentiated value for those elements is -1/32 or -1/n
# and for the rest of the elements in the logprobs, its zero as they are not picked, that means there impact on loss is zero

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n

cmp('logprobs', dlogprobs, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


In [51]:
dprobs = dlogprobs * (1/probs)
cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


In [52]:
counts_sum_inv.shape

torch.Size([32, 1])

In [53]:
# dcounts_sum_inv has to be the size of counts_sum_inv, but counts_sum_inv is stretched to match the dimension of counts
# due to that we have to sum all the derivatives and go back
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
dcounts_sum_inv.shape

counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0


torch.Size([32, 1])

In [60]:
dcounts = counts_sum_inv * dprobs # this is wrong, as we have 2 paths to counts from probs, one directly and one through counts_sum_inv
# so we have to calculate both and add them to get the correct result
# so we will calculate the other path first
cmp('counts', dcounts, counts)
dcounts.shape

counts          | exact: False | approximate: False | maxdiff: 0.006552322302013636


torch.Size([32, 27])

In [55]:
dcounts_sum_inv.shape, counts_sum.shape

(torch.Size([32, 1]), torch.Size([32, 1]))

In [57]:
dcounts_sum = - dcounts_sum_inv * counts_sum**-2
cmp('counts_sum', dcounts_sum, counts_sum)

counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0


In [58]:
counts_sum.shape, counts.shape

(torch.Size([32, 1]), torch.Size([32, 27]))

In [63]:
# so dcounts will have 2 parts, and we add them, the dcount_sums path will be 1 as its the derivative of itself and then we add the counts_sum_inv * counts part
dcounts = dcounts_sum * torch.ones((32, 27)) + counts_sum_inv * dprobs
cmp('counts', dcounts, counts)

counts          | exact: True  | approximate: True  | maxdiff: 0.0


In [64]:
dnorm_logits = dcounts * counts
cmp('dnorm_logits', dnorm_logits, norm_logits)

dnorm_logits    | exact: True  | approximate: True  | maxdiff: 0.0


In [66]:
logit_maxes.shape, dnorm_logits.shape

(torch.Size([32, 1]), torch.Size([32, 27]))

In [68]:
dlogit_maxes = (dnorm_logits * -1).sum(1, keepdim=True) # we sum as this was stretched from 32,1 to 32, 27, so we squeeze back by adding
cmp('dlogit_maxes', dlogit_maxes, logit_maxes)

dlogit_maxes    | exact: True  | approximate: True  | maxdiff: 0.0


In [71]:
logits.shape, logit_maxes.shape

(torch.Size([32, 27]), torch.Size([32, 1]))

In [77]:
# here we are taking max of tensor, so the element chosen has impact that is its derivative is 1, others derivative is zero
im = torch.zeros_like(logits)
max_indices = torch.argmax(logits, dim=1, keepdim=True)
im.scatter_(1, max_indices, 1)

dlogits = dnorm_logits * 1.0 + dlogit_maxes * im
cmp('dlogits', dlogits, logits)

dlogits         | exact: True  | approximate: True  | maxdiff: 0.0


In [78]:
dh = dlogits @ W2.T
cmp('dh', dh, h)

h               | exact: True  | approximate: True  | maxdiff: 0.0


In [79]:
dW2 = h.T @ dlogits
cmp('dW2', dW2, W2)

W2              | exact: True  | approximate: True  | maxdiff: 0.0


In [80]:
db2 = dlogits.sum(0)
cmp('db2', db2, b2)

b2              | exact: True  | approximate: True  | maxdiff: 0.0


In [83]:
dhpreact = dh * (1 - h**2)
cmp('dhpreact', dhpreact, hpreact)

dhpreact        | exact: True  | approximate: True  | maxdiff: 0.0


In [86]:
bngain.shape, dhpreact.shape, bnraw.shape

(torch.Size([1, 64]), torch.Size([32, 64]), torch.Size([32, 64]))

In [90]:
dbngain = (dhpreact * bnraw).sum(0, keepdim = True)
cmp('dbngain', dbngain, bngain)

dbngain         | exact: True  | approximate: True  | maxdiff: 0.0


In [91]:
dbnraw = dhpreact * bngain
cmp('dbnraw', dbnraw, bnraw)

dbnraw          | exact: True  | approximate: True  | maxdiff: 0.0


In [93]:
bnbias.shape, dhpreact.shape

(torch.Size([1, 64]), torch.Size([32, 64]))

In [95]:
dbnbias = (dhpreact * 1.0).sum(0, keepdim=True)
cmp('dbnbias', dbnbias, bnbias)

dbnbias         | exact: True  | approximate: True  | maxdiff: 0.0


In [98]:
# bnraw = bndiff * bnvar_inv
bnraw.shape, bndiff.shape, bnvar_inv.shape

(torch.Size([32, 64]), torch.Size([32, 64]), torch.Size([1, 64]))

In [100]:
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
cmp('dbnvar_inv', dbnvar_inv, bnvar_inv)

dbnvar_inv      | exact: True  | approximate: True  | maxdiff: 0.0


In [101]:
dbndiff = dbnraw * bnvar_inv
cmp('dbndiff', dbndiff, bndiff) # this is wrong because there are 2 path to bndiff, we will add the other part later and it will be correct

dbndiff         | exact: False | approximate: False | maxdiff: 0.0010155793279409409


In [102]:
#bnvar_inv = (bnvar + 1e-5)**-0.5
dbnvar_inv.shape, bnvar.shape

(torch.Size([1, 64]), torch.Size([1, 64]))

In [105]:
dbnvar = dbnvar_inv * (-0.5*(bnvar+1e-5)**-1.5)
cmp('dbnvar', dbnvar, bnvar)

dbnvar          | exact: True  | approximate: True  | maxdiff: 0.0


In [106]:
#bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
bnvar.shape, bndiff2.shape, dbnvar.shape

(torch.Size([1, 64]), torch.Size([32, 64]), torch.Size([1, 64]))

In [113]:
dbndiff2 = 1/(n-1)*torch.ones_like(bndiff2) * dbnvar
cmp('dbndiff2', dbndiff2, bndiff2)

dbndiff2        | exact: True  | approximate: True  | maxdiff: 0.0


In [114]:
#bndiff2 = bndiff**2
dbndiff += dbndiff2 * 2 * bndiff
cmp('dbndiff', dbndiff, bndiff)

dbndiff         | exact: True  | approximate: True  | maxdiff: 0.0


In [116]:
#bndiff = hprebn - bnmeani
hprebn.shape, bnmeani.shape, dbndiff.shape

(torch.Size([32, 64]), torch.Size([1, 64]), torch.Size([32, 64]))

In [117]:
dbnmeani = (dbndiff * -1.0).sum(0, keepdim=True)
cmp('dbmeani', dbnmeani, bnmeani)

dbmeani         | exact: True  | approximate: True  | maxdiff: 0.0


In [118]:
dhprebn = dbndiff * 1.0
cmp('dhprebn', dhprebn, hprebn) #is wrong bc hprebn has 2 path

dhprebn         | exact: False | approximate: False | maxdiff: 0.0010734996758401394


In [119]:
#bnmeani = 1/n*hprebn.sum(0, keepdim=True)
dhprebn += 1/n*torch.ones_like(hprebn) * dbnmeani
cmp('dhprebn', dhprebn, hprebn)

dhprebn         | exact: True  | approximate: True  | maxdiff: 0.0


In [120]:
#hprebn = embcat @ W1 + b1
hprebn.shape, embcat.shape, W1.shape, b1.shape

(torch.Size([32, 64]),
 torch.Size([32, 30]),
 torch.Size([30, 64]),
 torch.Size([64]))

In [121]:
dembcat = dhprebn @ W1.T
cmp('dembcat', dembcat, embcat)

dembcat         | exact: True  | approximate: True  | maxdiff: 0.0


In [122]:
dW1 = embcat.T @ dhprebn
cmp('dW1', dW1, W1)

dW1             | exact: True  | approximate: True  | maxdiff: 0.0


In [123]:
db1 = dhprebn.sum(0)
cmp('db1', db1, b1)

db1             | exact: True  | approximate: True  | maxdiff: 0.0


In [124]:
demb = dembcat.view(emb.shape)
cmp('demb', demb, emb)

demb            | exact: True  | approximate: True  | maxdiff: 0.0


In [125]:
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]
cmp('dC', dC, C)

dC              | exact: True  | approximate: True  | maxdiff: 0.0


In [126]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = counts * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h**2) * dh
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff += (2*bndiff) * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]
    
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [127]:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3322417736053467 diff: -4.76837158203125e-07


In [128]:
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 6.05359673500061e-09
